In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv('data.csv')
df.head()
print(df.shape)
print(df.info(verbose=True))
df.isnull().sum()

total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")

X = df.drop(columns=['Target'])
y = df['Target']
le = LabelEncoder()
y = le.fit_transform(y)
X = pd.get_dummies(X , columns= ['Course'], drop_first=True)

#Feature Scaling 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train , X_test , y_train , y_test = train_test_split(X_scaled , y , test_size = 0.2 , random_state=42)
#Handle Imbalance data 

sm = SMOTE()
X_train , y_train = sm.fit_resample(X_train , y_train)



model = RandomForestClassifier(random_state = 42)
model.fit(X_train , y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
#Hyper Parameter Tuning 
#Optimize the models performance by tuning hyper parameters . This will help find the 
#best configuration for better accuracy and generalization in predicting student 
#academic risk 


#Step 1 : Set up MLFLOW for experiment tracking 
#Step 2 : Perform Hyper Parameter Tuning 
#Step 3 : Log Results to MLFLOW 

#Step 1 
import mlflow
mlflow.set_experiment("Student_Risk_Prediction")

#step 2 
# Use grid search to find the best parameter for model 

param_grid = {
    'n_estimators' : [100 , 200],
    'max_depth' : [10 , 20 , None],
    'min_samples_split' : [2 , 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state = 42),
                    param_grid = param_grid  , 
                    cv = 3 , 
                    scoring = 'accuracy',
                    n_jobs =-1
                    )

grid.fit(X_train , y_train)

#step 3 

with mlflow.start_run():
    mlflow.log_params(grid.best_params_)
    mlflow.log_metric("best accuracy" , grid.best_score_)
    mlflow.sklearn.log_model(grid.best_estimator_ , "model")


#Model Evaluation 
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
#Serialize the trained model 

import joblib
joblib.dump(best_model , "model.pkl")


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate
